# Tutorial 19: how `universal-geometry-v2` works, one design at a time

Tutorial 18 showed **that** v2 outperforms `static-shape-v0`. This tutorial shows
**how** it is computed. We take a single real capacitor out of the catalogue and
follow it all the way from GDS polygons to a 512-number vector, drawing every
intermediate quantity.

The encoder is deliberately simple enough to audit. Every coordinate is a
physical measurement, so each step below can be checked against a picture of the
geometry rather than against a training loss.

```
GDS polygons
  -> layer roles and terminals
  -> block 1  physical metrics        (48)
  -> block 2  coupling spectrum      (192)
  -> block 3  shape spectrum         (128)
  -> block 4  parameter statistics    (96)
  -> block 5  physics proxy           (48)
  -> 512-dimensional vector
```

Along the way we will **re-derive the coupling spectrum by hand** with plain
`shapely` and `numpy`, and check that it reproduces the library output exactly.
The final section runs the scenario the whole design exists for: a stranger's
capacitor, with its own shape and its own 28 parameter names, projected into the
space our catalogue already occupies.

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import shapely
from huggingface_hub import hf_hub_download
from plotly.subplots import make_subplots
from scipy import ndimage

from squadds.layouts.geometry_v2 import (
    COUPLING_BINS,
    COUPLING_BLOCK_SIZE,
    COUPLING_EDGES,
    GROUND_BINS,
    METRIC_BLOCK_SIZE,
    METRIC_NAMES,
    PARAMETER_BLOCK_SIZE,
    PARAMETER_DIMENSION_CLASSES,
    PHYSICS_BLOCK_SIZE,
    SHAPE_BINS,
    SHAPE_BLOCK_SIZE,
    SHAPE_EDGES,
    TOPOLOGY_RADII_UM,
    V2_DIMENSIONS,
    VACUUM_PERMITTIVITY,
    classify_parameter,
    encode,
    parameter_block,
    read_layer_geometry,
    soft_histogram,
)
from squadds.layouts.geometry_v2 import (
    _boundary_samples,
    _raster_frame,
    _rasterize,
    _role_geometry,
    _terminals,
)

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 30)

INK = "#1F2937"
ROLE_COLORS = {
    "conductor": "#00798C",
    "etch": "#E9C46A",
    "port": "#D1495B",
    "domain": "#C7CDD4",
}
TERMINAL_COLORS = ["#00798C", "#D1495B", "#6A4C93", "#E9C46A"]

SOURCE_ID = "exp6/cap_0000"
GDS_PATH = Path(
    hf_hub_download(
        "SQuADDS/SQuADDS_Layouts",
        f"raw/GeneralizedCapNInterdigital/{SOURCE_ID}.gds",
        repo_type="dataset",
    )
)
DATABASE = Path(
    hf_hub_download(
        "SQuADDS/SQuADDS_DB",
        "coupler-GeneralizedCapNInterdigital-cap_matrix.json",
        repo_type="dataset",
    )
)
row = next(item for item in json.loads(DATABASE.read_text()) if item["notes"]["source_id"] == SOURCE_ID)
DESIGN_OPTIONS = row["design"]["design_options"]
MEASURED = row["sim_results"]

print(f"worked example : {SOURCE_ID}")
print(f"design options : {len(DESIGN_OPTIONS)} entries")
print(f"simulated C(N,S) = {MEASURED['north_to_south']:.4f} fF")

/Users/shanto/LFL/fall26/SQuADDS/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/SQuADDS/SQuADDS_Layouts/resolve/main/raw/GeneralizedCapNInterdigital/exp6/cap_0000.gds "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/SQuADDS/SQuADDS_Layouts/312357d4c4f90265f4fc431b9d2f13563306939d/raw%2FGeneralizedCapNInterdigital%2Fexp6%2Fcap_0000.gds "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/SQuADDS/SQuADDS_DB/resolve/main/coupler-GeneralizedCapNInterdigital-cap_matrix.json "HTTP/1.1 302 Found"


worked example : exp6/cap_0000
design options : 41 entries
simulated C(N,S) = 0.7104 fF


### The pipeline at a glance

Every step below is a measurement with an input, an operation, and a fixed slice
of the output vector. Nothing is fitted and nothing depends on any other design.

| # | Step | Reads | Produces | Coordinates |
| --- | --- | --- | --- | --- |
| 1 | Layer roles | `(layer, datatype)` pairs | conductor / etch / port / domain sets | - |
| 2 | Terminals | conductor set | ordered list of connected components | - |
| 3 | Physical metrics | terminals, raster | 48 named scalars in um | 0-47 |
| 4 | Coupling spectrum | boundary samples, exact distances | facing length per separation bin | 48-239 |
| 5 | Shape spectrum | raster FFT, contour FFT | correlations, widths, harmonics | 240-367 |
| 6 | Parameter statistics | the option mapping | dimension-typed order statistics | 368-463 |
| 7 | Physics proxy | boundary-element solve, dilation | 2D capacitance matrix, topology | 464-511 |

Follow the coordinate column. By the end of section 8 every one of the 512 slots
will have been accounted for, and section 8 replays the whole thing as a single
interactive figure.

## 1. The input contract

v2 takes exactly two things: a GDS file whose `(layer, datatype)` pairs follow
the published semantics, and the native design-parameter mapping from whatever
tool produced it. Nothing else. No catalogue, no fitted statistics, no
simulation results.

The four roles the encoder recognizes are `conductor`, `etch`, `port`, and
`domain`. For this component family the mapping is:

| layer | datatype | role |
| --- | --- | --- |
| 1 | 10 | conductor - the signal metal |
| 1 | 0 | domain - the ground plane, with a cutout around the device |
| 2 | 0 | port - marks the north terminal |
| 3 | 0 | port - marks the south terminal |

In [2]:
geometry = read_layer_geometry(GDS_PATH)
grouped = _role_geometry(geometry, None)

layer_table = pd.DataFrame(
    [
        {
            "layer": key[0],
            "datatype": key[1],
            "role": role,
            "polygons": len(list(getattr(shape, "geoms", [shape]))),
            "area_um2": round(shape.area, 3),
            "perimeter_um": round(shape.length, 3),
        }
        for role, entries in grouped.items()
        for key, shape in entries
    ]
).sort_values(["layer", "datatype"])
print(layer_table.to_string(index=False))
print()
print("design options (first 8 of %d):" % len(DESIGN_OPTIONS))
for name in sorted(DESIGN_OPTIONS)[:8]:
    print(f"  {name:32s} {DESIGN_OPTIONS[name]}")

 layer  datatype      role  polygons  area_um2  perimeter_um
     1         0    domain         1 31478.400       820.562
     1        10 conductor         2   185.541       104.140
     2         0      port         1     4.400         8.400
     3         0      port         1     4.400         8.400

design options (first 8 of 41):
  chip                             main
  east_gap_ground                  4um
  finger_count                     2
  finger_edge_gap_radius           1um
  finger_etch_radius               5.0um
  finger_gap_east_west             3um
  finger_gap_north_south           3um
  finger_length                    8um


In [3]:
# %% hide input
def polygon_traces(shape, name, color, *, opacity=0.75, paper="#FFFFFF", show=True):
    traces = []
    first = True
    for polygon in getattr(shape, "geoms", [shape]):
        if polygon.geom_type != "Polygon":
            continue
        x, y = polygon.exterior.xy
        traces.append(
            go.Scatter(
                x=list(x),
                y=list(y),
                mode="lines",
                fill="toself",
                fillcolor=color,
                opacity=opacity,
                line={"color": color, "width": 1.2},
                name=name,
                legendgroup=name,
                showlegend=show and first,
                hoverinfo="skip",
            )
        )
        first = False
        for interior in polygon.interiors:
            hx, hy = interior.xy
            traces.append(
                go.Scatter(
                    x=list(hx),
                    y=list(hy),
                    mode="lines",
                    fill="toself",
                    fillcolor=paper,
                    line={"color": color, "width": 0.8},
                    legendgroup=name,
                    showlegend=False,
                    hoverinfo="skip",
                )
            )
    return traces


figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Full GDS, all four roles", "Functional geometry the encoder measures"],
    horizontal_spacing=0.09,
)
for role in ("domain", "etch", "conductor", "port"):
    for key, shape in grouped[role]:
        for trace in polygon_traces(shape, role, ROLE_COLORS[role]):
            figure.add_trace(trace, row=1, col=1)
for role in ("conductor", "port"):
    for key, shape in grouped[role]:
        for trace in polygon_traces(shape, role, ROLE_COLORS[role], show=False):
            figure.add_trace(trace, row=1, col=2)
figure.update_yaxes(scaleanchor="x", scaleratio=1, row=1, col=1)
figure.update_yaxes(scaleanchor="x2", scaleratio=1, row=1, col=2)
figure.update_xaxes(title_text="x (um)")
figure.update_yaxes(title_text="y (um)", row=1, col=1)
figure.update_layout(
    title=f"{SOURCE_ID}: a two-terminal interdigital capacitor",
    template="plotly_white",
    height=520,
)
figure.show()

The left panel is everything in the file. The ground plane dominates it, and
notice that it is **not** a solid rectangle: it has a cutout around the device.
That cutout is a real physical gap, and the encoder measures it.

The right panel is what remains after the ground plane is set aside as context.
This is the functional geometry.

## 2. Step one: terminals are discovered, not declared

A design tool knows that this component has a "north pad" and a "south pad". The
encoder must not, because a foreign contributor will use different words.

Instead v2 merges the conductor layer and splits it into **connected
components**. Each component is a terminal. Ordering is then fixed by two
deterministic rules:

1. if port markers exist, order terminals by the port layer nearest to each;
2. otherwise, order by descending area, breaking ties by first appearance.

This is what makes `terminal 0` and `terminal 1` mean the same thing in a
28-parameter foreign layout as in this one.

In [4]:
conductor = shapely.union_all([shape for _, shape in grouped["conductor"]])
terminals = _terminals(conductor, grouped["port"])

terminal_table = pd.DataFrame(
    [
        {
            "terminal": index,
            "area_um2": round(terminal.area, 3),
            "perimeter_um": round(terminal.length, 3),
            "nearest port layer": min(
                (key[0] for key, port in grouped["port"] if terminal.distance(port) < 1e-6),
                default=None,
            ),
        }
        for index, terminal in enumerate(terminals)
    ]
)
print(terminal_table.to_string(index=False))
print()
print(f"minimum separation between terminal 0 and terminal 1: {terminals[0].distance(terminals[1]):.4f} um")

 terminal  area_um2  perimeter_um  nearest port layer
        0    92.771         52.07                   2
        1    92.771         52.07                   3

minimum separation between terminal 0 and terminal 1: 2.9956 um


In [5]:
# %% hide input
figure = go.Figure()
for key, shape in grouped["domain"]:
    for trace in polygon_traces(shape, "ground plane", ROLE_COLORS["domain"], opacity=0.35):
        figure.add_trace(trace)
for index, terminal in enumerate(terminals):
    for trace in polygon_traces(terminal, f"terminal {index}", TERMINAL_COLORS[index]):
        figure.add_trace(trace)
for key, port in grouped["port"]:
    for trace in polygon_traces(port, f"port layer {key[0]}", ROLE_COLORS["port"], opacity=1.0):
        figure.add_trace(trace)
figure.update_yaxes(scaleanchor="x", scaleratio=1)
figure.update_layout(
    title="Connected components become terminals; port markers fix their order",
    xaxis_title="x (um)",
    yaxis_title="y (um)",
    template="plotly_white",
    height=560,
)
figure.show()

## 3. Step two: the coupling spectrum

This is the block the design rests on, so we derive it from scratch.

For coplanar conductors the mutual capacitance is approximately an integral of
some kernel over facing boundary length at a given separation,

$$C_{ij} \;\approx\; \varepsilon \int f(d)\, \mathrm{d}L .$$

We do not know $f$, and we do not need to. If we store $\mathrm{d}L$ **binned
against absolute separation $d$**, then any model that is linear in those bins
can represent any $f$. The learned head fits the kernel; the encoder supplies
the integral's measure.

Concretely, for terminal $i$ against terminal $j$:

1. walk terminal $i$'s boundary at uniform arclength, taking 1,024 samples;
2. for each sample, compute the exact distance to terminal $j$;
3. accumulate each sample's arclength share into a frozen log-spaced grid of 24
   bins spanning 0.1 um to 1000 um;
4. symmetrize over both directions and take `log1p`.

The bin edges are constants of the standard, never fitted, which is exactly what
makes one contributor's bin 12 the same physical quantity as another's.

In [6]:
north, south = terminals[0], terminals[1]


def sample_boundary(shape, count=1024):
    """Uniform-arclength samples with the boundary length each one represents."""
    boundary = shape.boundary
    length = boundary.length
    positions = (np.arange(count) + 0.5) * length / count
    coordinates = shapely.get_coordinates(shapely.line_interpolate_point(boundary, positions))
    return coordinates, np.full(count, length / count)


north_points, north_weights = sample_boundary(north)
south_points, south_weights = sample_boundary(south)

north_distances = shapely.distance(shapely.points(north_points), south)
south_distances = shapely.distance(shapely.points(south_points), north)

forward = soft_histogram(north_distances, north_weights, COUPLING_EDGES)
backward = soft_histogram(south_distances, south_weights, COUPLING_EDGES)
hand_derived = np.log1p(0.5 * (forward + backward))

vector = encode(GDS_PATH, DESIGN_OPTIONS)
library = vector[METRIC_BLOCK_SIZE : METRIC_BLOCK_SIZE + COUPLING_BINS]

print(f"largest disagreement between the hand derivation and encode(): {np.abs(hand_derived - library).max():.2e}")
assert np.allclose(hand_derived, library, atol=1e-5)

centers = np.sqrt(COUPLING_EDGES[:-1] * COUPLING_EDGES[1:])
spectrum_table = pd.DataFrame(
    {
        "bin": range(COUPLING_BINS),
        "separation_um": centers.round(3),
        "facing_boundary_um": (0.5 * (forward + backward)).round(4),
        "stored_value": hand_derived.round(4),
    }
)
print(spectrum_table[spectrum_table.facing_boundary_um > 0].to_string(index=False))

largest disagreement between the hand derivation and encode(): 8.36e-08
 bin  separation_um  facing_boundary_um  stored_value
   8          2.610              9.5775        2.3587
   9          3.831              8.2403        2.2236
  10          5.623              6.5737        2.0247
  11          8.254             15.9765        2.8318
  12         12.115             10.7259        2.4618
  13         17.783              0.9761        0.6811


In [7]:
# %% hide input
figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Every boundary sample, coloured by distance to the facing terminal",
        "The same distances, binned into the coupling spectrum",
    ],
    horizontal_spacing=0.11,
    column_widths=[0.46, 0.54],
)
for trace in polygon_traces(south, "terminal 1", TERMINAL_COLORS[1], opacity=0.25, show=False):
    figure.add_trace(trace, row=1, col=1)
for trace in polygon_traces(north, "terminal 0", TERMINAL_COLORS[0], opacity=0.18, show=False):
    figure.add_trace(trace, row=1, col=1)
figure.add_trace(
    go.Scatter(
        x=north_points[:, 0],
        y=north_points[:, 1],
        mode="markers",
        marker={
            "size": 5,
            "color": north_distances,
            "colorscale": "Viridis",
            "cmin": 0,
            "cmax": float(np.quantile(north_distances, 0.95)),
            "colorbar": {"title": "distance<br>to facing<br>terminal (um)", "x": 0.43, "len": 0.85},
        },
        name="terminal 0 boundary",
        showlegend=False,
        hovertemplate="x=%{x:.2f} um<br>y=%{y:.2f} um<br>distance=%{marker.color:.2f} um<extra></extra>",
    ),
    row=1,
    col=1,
)
figure.add_trace(
    go.Bar(
        x=centers,
        y=0.5 * (forward + backward),
        marker_color="#00798C",
        name="facing boundary length",
        showlegend=False,
        hovertemplate="separation=%{x:.2f} um<br>boundary length=%{y:.3f} um<extra></extra>",
    ),
    row=1,
    col=2,
)
figure.update_yaxes(scaleanchor="x", scaleratio=1, row=1, col=1)
figure.update_xaxes(title_text="x (um)", row=1, col=1)
figure.update_yaxes(title_text="y (um)", row=1, col=1)
figure.update_xaxes(title_text="conductor separation (um)", type="log", row=1, col=2)
figure.update_yaxes(title_text="facing boundary length (um)", row=1, col=2)
figure.update_layout(
    title="From geometry to spectrum: where the metal faces metal, and how far away",
    template="plotly_white",
    height=520,
)
figure.show()

The left panel is the whole idea in one picture. The dark points along the inner
edges of the fingers sit a couple of micrometers from the facing comb and carry
almost all of the capacitance; the bright points on the outer edges are tens of
micrometers away and contribute little. The right panel is that same information
with the geometry thrown away and only the physics kept.

This also explains why the minimum gap on its own is a poor predictor. It tells
you where the leftmost bar sits, but not how tall it is - and the height, the
facing boundary length, is what multiplies the kernel.

### Why the bins are soft

A hard histogram assigns each sample entirely to one bin. A sample sitting a
hair from a bin edge then jumps its whole weight to the neighbour when the
geometry moves by a nanometre, which makes the embedding discontinuous in the
thing it is supposed to measure. v2 splits each sample linearly between the two
nearest bin centres in log-distance, so the spectrum is Lipschitz in the
geometry.

In [8]:
# %% hide input
probe_edges = COUPLING_EDGES
probe_centers = centers
shift_grid = np.linspace(0.0, 0.35, 60)
hard_track, soft_track = [], []
for shift in shift_grid:
    shifted = north_distances * (1.0 + shift)
    hard, _ = np.histogram(np.clip(shifted, probe_edges[0], probe_edges[-1]), bins=probe_edges, weights=north_weights)
    soft = soft_histogram(shifted, north_weights, probe_edges)
    hard_track.append(hard)
    soft_track.append(soft)
hard_track = np.asarray(hard_track)
soft_track = np.asarray(soft_track)
busiest = int(np.argmax(soft_track[0]))

figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        f"Bin {busiest} as the geometry is stretched",
        "Total change in the whole spectrum",
    ],
    horizontal_spacing=0.12,
)
figure.add_trace(
    go.Scatter(
        x=100 * shift_grid,
        y=hard_track[:, busiest],
        mode="lines",
        name="hard bins",
        line={"color": "#D1495B", "width": 3},
    ),
    row=1,
    col=1,
)
figure.add_trace(
    go.Scatter(
        x=100 * shift_grid,
        y=soft_track[:, busiest],
        mode="lines",
        name="soft bins (v2)",
        line={"color": "#00798C", "width": 3},
    ),
    row=1,
    col=1,
)
figure.add_trace(
    go.Scatter(
        x=100 * shift_grid[1:],
        y=np.abs(np.diff(hard_track, axis=0)).sum(axis=1),
        mode="lines",
        name="hard bins",
        line={"color": "#D1495B", "width": 3},
        showlegend=False,
    ),
    row=1,
    col=2,
)
figure.add_trace(
    go.Scatter(
        x=100 * shift_grid[1:],
        y=np.abs(np.diff(soft_track, axis=0)).sum(axis=1),
        mode="lines",
        name="soft bins (v2)",
        line={"color": "#00798C", "width": 3},
        showlegend=False,
    ),
    row=1,
    col=2,
)
figure.update_xaxes(title_text="all separations stretched by (%)")
figure.update_yaxes(title_text="boundary length in the bin (um)", row=1, col=1)
figure.update_yaxes(title_text="step-to-step change (um)", row=1, col=2)
figure.update_layout(
    title="Hard bins jump; soft bins move smoothly with the geometry",
    template="plotly_white",
    height=470,
)
figure.show()

The staircase on the left is a hard histogram handing an entire bin's worth of
boundary to its neighbour in one step. The right panel shows the same effect as
total spectrum movement: the hard version spikes whenever a population crosses
an edge, while the soft version stays bounded.

The remaining 168 coordinates of this block cover the other five terminal pairs
and each terminal's spectrum against the ground plane. For a two-terminal device
most stay zero, and that reserved capacity is what lets a three- or four-terminal
component use the same 512 coordinates.

## 4. Step three: physical metrics

Forty-eight scalars, each a named measurement in absolute units. The point of
this block is that a human can read it, and every entry has a dimension. Below
are the twelve that matter most for this device, next to their raw values.

In [9]:
frame = _raster_frame(conductor.bounds)
pixel = frame[2]
conductor_mask = _rasterize(conductor, frame)
interior = ndimage.distance_transform_edt(conductor_mask) * pixel
widths = 2.0 * interior[conductor_mask]

highlights = [
    "log1p_bbox_width_um",
    "log1p_bbox_height_um",
    "log1p_conductor_area_um2",
    "log1p_conductor_perimeter_um",
    "log1p_minimum_pair_gap_um",
    "log1p_primary_inverse_gap_integral",
    "log1p_conductor_width_p05_um",
    "log1p_conductor_width_p50_um",
    "terminal_count",
    "horizontal_symmetry",
    "vertical_symmetry",
    "conductor_fill_fraction",
]
metric_table = pd.DataFrame(
    [
        {
            "metric": name,
            "stored value": round(float(vector[METRIC_NAMES.index(name)]), 4),
            "physical value": (
                round(float(np.expm1(vector[METRIC_NAMES.index(name)])), 4)
                if name.startswith("log1p_")
                else round(float(vector[METRIC_NAMES.index(name)]), 4)
            ),
        }
        for name in highlights
    ]
)
print(metric_table.to_string(index=False))
print()
print(f"conductor width percentiles from the distance transform (um): {np.percentile(widths, [5, 50, 95]).round(3)}")

                            metric  stored value  physical value
               log1p_bbox_width_um        2.4849         11.0000
              log1p_bbox_height_um        3.4657         31.0000
          log1p_conductor_area_um2        5.2287        185.5414
      log1p_conductor_perimeter_um        4.6553        104.1401
         log1p_minimum_pair_gap_um        1.3852          2.9956
log1p_primary_inverse_gap_integral        3.0176         19.4427
      log1p_conductor_width_p05_um        0.2362          0.2664
      log1p_conductor_width_p50_um        1.1414          2.1312
                    terminal_count        2.0000          2.0000
               horizontal_symmetry        0.9441          0.9441
                 vertical_symmetry        0.9441          0.9441
           conductor_fill_fraction        0.5441          0.5441

conductor width percentiles from the distance transform (um): [0.266 2.131 5.062]


In [10]:
# %% hide input
extent_x = [frame[0], frame[0] + frame[3]]
extent_y = [frame[1], frame[1] + frame[3]]
figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Distance to the nearest conductor edge", "Conductor width distribution"],
    horizontal_spacing=0.13,
)
figure.add_trace(
    go.Heatmap(
        z=np.where(conductor_mask, interior, np.nan),
        x=np.linspace(extent_x[0], extent_x[1], conductor_mask.shape[1]),
        y=np.linspace(extent_y[1], extent_y[0], conductor_mask.shape[0]),
        colorscale="Magma",
        colorbar={"title": "half-width<br>(um)", "x": 0.43, "len": 0.85},
        hovertemplate="x=%{x:.2f} um<br>y=%{y:.2f} um<br>half-width=%{z:.2f} um<extra></extra>",
    ),
    row=1,
    col=1,
)
figure.add_trace(
    go.Bar(
        x=np.sqrt(SHAPE_EDGES[:-1] * SHAPE_EDGES[1:]),
        y=soft_histogram(widths, np.full(len(widths), pixel * pixel), SHAPE_EDGES),
        marker_color="#6A4C93",
        showlegend=False,
        hovertemplate="width=%{x:.2f} um<br>area=%{y:.2f} um^2<extra></extra>",
    ),
    row=1,
    col=2,
)
figure.update_yaxes(scaleanchor="x", scaleratio=1, autorange="reversed", row=1, col=1)
figure.update_xaxes(title_text="x (um)", row=1, col=1)
figure.update_xaxes(title_text="conductor width (um)", type="log", row=1, col=2)
figure.update_yaxes(title_text="conductor area at that width (um^2)", row=1, col=2)
figure.update_layout(
    title="Feature size is measured, not inferred from a parameter name",
    template="plotly_white",
    height=500,
)
figure.show()

The bright ridge running down each finger is its half-width. Reducing that map
to percentiles gives minimum feature size, typical linewidth, and pad size as
three absolute numbers, none of which require knowing what the design tool
called them. A contributor whose parameter is named `digit_thickness` produces
the same coordinates as ours named `finger_width`.

## 5. Step four: the shape spectrum

Four channels of 32 bins. Two are correlation functions computed by FFT on a
raster whose pixel size is recorded, so the radial bins stay in micrometers:

- **two-point correlation** of the conductor set, the probability that two points
  separated by $r$ are both metal, which reveals the finger pitch;
- **cross-correlation** between terminal 0 and terminal 1, which reveals the
  interleaving;
- the **width distribution** shown above;
- **contour harmonics**, the Fourier transform of the boundary traversed at
  uniform arclength, taken on exact polygon vertices with no rasterization.

In [11]:
# %% hide input
def autocorrelate(first, second):
    spectrum = np.fft.rfft2(first.astype(float))
    other = spectrum if second is first else np.fft.rfft2(second.astype(float))
    return np.fft.fftshift(np.fft.irfft2(spectrum * np.conj(other), s=first.shape)) / first.size


def radial(correlation, pixel_size, edges):
    size = correlation.shape[0]
    grid = np.arange(size) - size // 2
    yy, xx = np.meshgrid(grid, grid, indexing="ij")
    radius = np.hypot(xx, yy) * pixel_size
    index = np.digitize(radius.reshape(-1), edges) - 1
    profile = np.zeros(len(edges) - 1)
    counts = np.zeros(len(edges) - 1)
    values = correlation.reshape(-1)
    valid = (index >= 0) & (index < len(profile))
    np.add.at(profile, index[valid], values[valid])
    np.add.at(counts, index[valid], 1.0)
    return profile / np.maximum(counts, 1.0)


masks = [_rasterize(terminal, frame) for terminal in terminals]
shape_centers = np.sqrt(SHAPE_EDGES[:-1] * SHAPE_EDGES[1:])
two_point = radial(autocorrelate(conductor_mask, conductor_mask), pixel, SHAPE_EDGES) / max(conductor_mask.mean(), 1e-12)
cross = radial(autocorrelate(masks[0], masks[1]), pixel, SHAPE_EDGES) / np.sqrt(
    max(masks[0].mean(), 1e-12) * max(masks[1].mean(), 1e-12)
)

contour, _ = _boundary_samples(terminals[0], target=256)
signal = contour[:, 0] + 1j * contour[:, 1]
spectrum = np.fft.fft(signal - signal.mean())

figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Correlation functions in absolute micrometers", "Contour rebuilt from N harmonics"],
    horizontal_spacing=0.12,
)
figure.add_trace(
    go.Scatter(
        x=shape_centers,
        y=two_point,
        mode="lines+markers",
        name="conductor with itself",
        line={"color": "#00798C", "width": 3},
    ),
    row=1,
    col=1,
)
figure.add_trace(
    go.Scatter(
        x=shape_centers,
        y=cross,
        mode="lines+markers",
        name="terminal 0 with terminal 1",
        line={"color": "#D1495B", "width": 3},
    ),
    row=1,
    col=1,
)
figure.add_trace(
    go.Scatter(
        x=contour[:, 0],
        y=contour[:, 1],
        mode="lines",
        name="exact contour",
        line={"color": INK, "width": 3},
    ),
    row=1,
    col=2,
)
for harmonics, color in ((4, "#E9C46A"), (8, "#6A4C93"), (16, "#00798C")):
    filtered = np.zeros_like(spectrum)
    filtered[:harmonics] = spectrum[:harmonics]
    filtered[-harmonics:] = spectrum[-harmonics:]
    rebuilt = np.fft.ifft(filtered) + signal.mean()
    figure.add_trace(
        go.Scatter(
            x=rebuilt.real,
            y=rebuilt.imag,
            mode="lines",
            name=f"{harmonics} harmonics",
            line={"color": color, "width": 2, "dash": "dash"},
        ),
        row=1,
        col=2,
    )
figure.update_xaxes(title_text="separation r (um)", type="log", row=1, col=1)
figure.update_yaxes(title_text="normalized correlation", row=1, col=1)
figure.update_xaxes(title_text="x (um)", row=1, col=2)
figure.update_yaxes(title_text="y (um)", scaleanchor="x2", scaleratio=1, row=1, col=2)
figure.update_layout(
    title="Shape measured as correlation length and as boundary harmonics",
    template="plotly_white",
    height=520,
)
figure.show()

On the right, four harmonics give the pad outline, eight begin to suggest the
comb, and sixteen resolve individual fingers. Storing the harmonic magnitudes
gives a compact, orientation-free description of exactly that progression - and
because it runs on polygon vertices, it never pays the resolution cost that sank
v0's 96 by 96 raster.

## 6. Step five: parameter statistics, without a name registry

This design has 40 options. A stranger's design will have a different number
with different names. v2 handles both with the same 96 coordinates by asking
what each parameter **is** rather than what it is called.

Each option is parsed into a physical dimension class - length, count, angle,
boolean, or other - using its unit suffix first and its name only as a fallback.
Within each class the encoder stores summary statistics plus **order
statistics**: the sorted smallest and largest values. `min(lengths)` is the
minimum feature size in the design, which is physically comparable across every
contributor, whatever they call it.

In [12]:
classified = []
for name, value in sorted(DESIGN_OPTIONS.items()):
    result = classify_parameter(name, value)
    if result is not None:
        classified.append({"parameter": name, "raw": str(value), "dimension": result[0], "canonical": result[1]})
classified_frame = pd.DataFrame(classified)

print(classified_frame["dimension"].value_counts().to_string())
print()
lengths = np.sort(classified_frame.query("dimension == 'length'")["canonical"].to_numpy())
print(f"the eight smallest lengths in this design (um): {lengths[:8].round(3)}")
print(f"the three largest  lengths in this design (um): {lengths[-3:].round(3)}")
print()
block, metadata = parameter_block(DESIGN_OPTIONS)
print(f"parameter block: {len(block)} dimensions from {metadata['parameter_count']} options")
print(f"identical to the block inside encode(): {np.allclose(block, vector[METRIC_BLOCK_SIZE + COUPLING_BLOCK_SIZE + SHAPE_BLOCK_SIZE : METRIC_BLOCK_SIZE + COUPLING_BLOCK_SIZE + SHAPE_BLOCK_SIZE + PARAMETER_BLOCK_SIZE], atol=1e-6)}")

dimension
length     35
count       2
boolean     2
angle       1

the eight smallest lengths in this design (um): [0. 0. 0. 0. 1. 1. 1. 1.]
the three largest  lengths in this design (um): [6. 6. 8.]

parameter block: 96 dimensions from 40 options
identical to the block inside encode(): True


In [13]:
# %% hide input
counts = classified_frame["dimension"].value_counts().reindex(PARAMETER_DIMENSION_CLASSES).fillna(0)
widths = {"length": 24, "count": 12, "angle": 8, "boolean": 4, "other": 12}
figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Options sorted into dimension classes", "The 24 length coordinates"],
    horizontal_spacing=0.13,
    specs=[[{"type": "bar"}, {"type": "bar"}]],
)
figure.add_trace(
    go.Bar(
        x=list(PARAMETER_DIMENSION_CLASSES),
        y=counts.to_numpy(),
        name="options",
        marker_color="#00798C",
        text=[int(value) for value in counts.to_numpy()],
        textposition="outside",
        hovertemplate="%{x}: %{y} options<extra></extra>",
    ),
    row=1,
    col=1,
)
figure.add_trace(
    go.Bar(
        x=list(PARAMETER_DIMENSION_CLASSES),
        y=[widths[name] for name in PARAMETER_DIMENSION_CLASSES],
        name="coordinates",
        marker_color="#E9C46A",
        text=[widths[name] for name in PARAMETER_DIMENSION_CLASSES],
        textposition="outside",
        hovertemplate="%{x}: %{y} coordinates<extra></extra>",
    ),
    row=1,
    col=1,
)
labels = ["count", "sum", "mean", "min", "max", "median", "sd", "range"] + [
    f"{index + 1}-smallest" for index in range(8)
] + [f"{index + 1}-largest" for index in range(8)]
figure.add_trace(
    go.Bar(
        x=labels,
        y=block[:24],
        marker_color="#6A4C93",
        showlegend=False,
        hovertemplate="%{x}<br>stored value=%{y:.3f}<extra></extra>",
    ),
    row=1,
    col=2,
)
figure.update_yaxes(title_text="number", row=1, col=1)
figure.update_xaxes(tickangle=-40, row=1, col=2)
figure.update_yaxes(title_text="stored value (signed log scale)", row=1, col=2)
figure.update_layout(
    title="Any parameter schema, any size, becomes the same 96 coordinates",
    template="plotly_white",
    barmode="group",
    height=520,
)
figure.show()

## 7. Step six: a physics proxy the model does not have to rediscover

The last block runs an actual, if crude, electrostatics calculation. The
conductor boundaries are discretized into segments, each pair gets the
two-dimensional free-space Green function

$$G_{mn} = -\frac{1}{2\pi\varepsilon_0}\ln\lVert r_m - r_n \rVert ,$$

and the linear system is solved for the charge that holds each terminal at unit
potential. Integrating that charge gives an approximate capacitance matrix.

This is **not** a simulation. It is two-dimensional, it ignores the substrate,
the finite metal thickness, and every three-dimensional fringing effect. That is
the point: it captures the part of the map that is identical for every component
class, so the learned head only has to fit a smoother, more transferable
correction instead of rediscovering the inverse-distance law from pixels.

In [14]:
segment_points, segment_lengths, owners = [], [], []
for index, terminal in enumerate(terminals[:4]):
    coordinates, weights = _boundary_samples(terminal, target=160)
    segment_points.append(coordinates)
    segment_lengths.append(weights)
    owners.append(np.full(len(coordinates), index))
points_matrix = np.vstack(segment_points)
segment = np.concatenate(segment_lengths)
labels_vector = np.concatenate(owners)

delta = points_matrix[:, None, :] - points_matrix[None, :, :]
separation = np.sqrt((delta**2).sum(axis=2)) * 1e-6
np.fill_diagonal(separation, 1.0)
green = -np.log(separation) / (2 * np.pi * VACUUM_PERMITTIVITY)
np.fill_diagonal(green, -(np.log(segment * 1e-6 / 2) - 1) / (2 * np.pi * VACUUM_PERMITTIVITY))
selector = np.stack([(labels_vector == index).astype(float) for index in range(len(terminals[:4]))], axis=1)
charges = np.linalg.solve(green, selector)
capacitance = selector.T @ charges

proxy_ff = np.abs(capacitance[0, 1]) * 1e15
print(f"two-dimensional proxy  |C(0,1)| = {proxy_ff:.4f} fF per metre of depth")
print(f"simulated (Q3D, 3D)     C(N,S)  = {MEASURED['north_to_south']:.4f} fF")
print()
print("These are different quantities in different units, so the numbers are not")
print("comparable and the proxy is not a prediction. What makes it a useful")
print("feature is that it is monotone in the simulated value across the whole")
print("catalogue: Tutorial 18 measures Spearman +0.920 between this coordinate")
print("and the Q3D mutual capacitance over all 13,683 designs.")

two-dimensional proxy  |C(0,1)| = 63148.4887 fF per metre of depth
simulated (Q3D, 3D)     C(N,S)  = 0.7104 fF

These are different quantities in different units, so the numbers are not
comparable and the proxy is not a prediction. What makes it a useful
feature is that it is monotone in the simulated value across the whole
catalogue: Tutorial 18 measures Spearman +0.920 between this coordinate
and the Q3D mutual capacitance over all 13,683 designs.


In [15]:
# %% hide input
figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Solved surface charge for terminal 0 held at unit potential",
        "Connected components and holes as the metal is dilated",
    ],
    horizontal_spacing=0.12,
)
density = charges[:, 0] / segment
limit = float(np.quantile(np.abs(density), 0.98))
figure.add_trace(
    go.Scatter(
        x=points_matrix[:, 0],
        y=points_matrix[:, 1],
        mode="markers",
        marker={
            "size": 7,
            "color": density,
            "colorscale": "RdBu",
            "cmid": 0,
            "cmin": -limit,
            "cmax": limit,
            "colorbar": {"title": "charge<br>density", "x": 0.43, "len": 0.85},
        },
        showlegend=False,
        hovertemplate="x=%{x:.2f} um<br>y=%{y:.2f} um<br>density=%{marker.color:.3g}<extra></extra>",
    ),
    row=1,
    col=1,
)
physics = vector[V2_DIMENSIONS - PHYSICS_BLOCK_SIZE :]
components = np.expm1(physics[16:32])
holes = np.expm1(physics[32:48])
figure.add_trace(
    go.Scatter(
        x=TOPOLOGY_RADII_UM,
        y=components,
        mode="lines+markers",
        name="connected components",
        line={"color": "#00798C", "width": 3},
    ),
    row=1,
    col=2,
)
figure.add_trace(
    go.Scatter(
        x=TOPOLOGY_RADII_UM,
        y=holes,
        mode="lines+markers",
        name="enclosed holes",
        line={"color": "#D1495B", "width": 3},
    ),
    row=1,
    col=2,
)
figure.update_yaxes(scaleanchor="x", scaleratio=1, row=1, col=1)
figure.update_xaxes(title_text="x (um)", row=1, col=1)
figure.update_yaxes(title_text="y (um)", row=1, col=1)
figure.update_xaxes(title_text="dilation radius (um)", type="log", row=1, col=2)
figure.update_yaxes(title_text="count", row=1, col=2)
figure.update_layout(
    title="A crude electrostatic solve, and the scale at which the combs merge",
    template="plotly_white",
    height=520,
)
figure.show()

Charge piles up on the facing finger edges and on the outer corners, which is
exactly where a textbook says it should. The right panel is the topology
signature: the two terminals stay separate until the dilation radius reaches
half the finger gap, then merge into one component. The radius at which that
happens **is** the gap, read off without ever naming it.

## 8. The whole algorithm in one interactive figure

Every piece has now been built separately. This figure replays them in order on
the same capacitor. Drag the slider: the left panel shows what the encoder is
looking at during that step, and the right panel shows which of the 512
coordinates that step writes, with every other coordinate greyed out.

Read it as the answer to "what does `encode` actually do". Each step consumes a
different view of the same geometry and writes a disjoint slice of the output.

In [16]:
# %% hide input
BLOCKS = [
    ("1-2. roles and terminals", None, None, "#C7CDD4"),
    ("3. physical metrics", 0, METRIC_BLOCK_SIZE, "#F4A261"),
    ("4. coupling spectrum", METRIC_BLOCK_SIZE, METRIC_BLOCK_SIZE + COUPLING_BLOCK_SIZE, "#00798C"),
    ("5. shape spectrum", METRIC_BLOCK_SIZE + COUPLING_BLOCK_SIZE,
     METRIC_BLOCK_SIZE + COUPLING_BLOCK_SIZE + SHAPE_BLOCK_SIZE, "#6A4C93"),
    ("6. parameter statistics", METRIC_BLOCK_SIZE + COUPLING_BLOCK_SIZE + SHAPE_BLOCK_SIZE,
     V2_DIMENSIONS - PHYSICS_BLOCK_SIZE, "#E9C46A"),
    ("7. physics proxy", V2_DIMENSIONS - PHYSICS_BLOCK_SIZE, V2_DIMENSIONS, "#2A9D8F"),
]

walk = make_subplots(
    rows=1, cols=2,
    subplot_titles=["what the encoder looks at", "which coordinates it writes"],
    horizontal_spacing=0.08, column_widths=[0.42, 0.58],
)
left_counts = []


def add_left(traces):
    for trace in traces:
        walk.add_trace(trace, row=1, col=1)
    left_counts.append(len(traces))


step_traces = []
for index, terminal in enumerate(terminals):
    step_traces.extend(polygon_traces(terminal, f"terminal {index}", TERMINAL_COLORS[index], show=False))
for _, port in grouped["port"]:
    step_traces.extend(polygon_traces(port, "port", ROLE_COLORS["port"], opacity=1.0, show=False))
add_left(step_traces)

add_left([
    go.Heatmap(
        z=np.where(conductor_mask, interior, np.nan),
        x=np.linspace(frame[0], frame[0] + frame[3], conductor_mask.shape[1]),
        y=np.linspace(frame[1] + frame[3], frame[1], conductor_mask.shape[0]),
        colorscale="Magma", showscale=False,
        hovertemplate="half-width=%{z:.2f} um<extra></extra>",
    )
])

add_left([
    go.Scatter(
        x=north_points[:, 0], y=north_points[:, 1], mode="markers",
        marker={"size": 5, "color": north_distances, "colorscale": "Viridis",
                "cmin": 0, "cmax": float(np.quantile(north_distances, 0.95))},
        hovertemplate="separation=%{marker.color:.2f} um<extra></extra>", showlegend=False,
    ),
    go.Scatter(
        x=south_points[:, 0], y=south_points[:, 1], mode="markers",
        marker={"size": 5, "color": south_distances, "colorscale": "Viridis",
                "cmin": 0, "cmax": float(np.quantile(north_distances, 0.95))},
        hovertemplate="separation=%{marker.color:.2f} um<extra></extra>", showlegend=False,
    ),
])

add_left([
    go.Heatmap(
        z=conductor_mask.astype(float),
        x=np.linspace(frame[0], frame[0] + frame[3], conductor_mask.shape[1]),
        y=np.linspace(frame[1] + frame[3], frame[1], conductor_mask.shape[0]),
        colorscale=[[0, "#FFFFFF"], [1, "#6A4C93"]], showscale=False, hoverinfo="skip",
    )
])

class_counts = classified_frame["dimension"].value_counts()
add_left([
    go.Bar(x=class_counts.index.tolist(), y=class_counts.to_numpy(), marker_color="#E9C46A",
           showlegend=False, hovertemplate="%{x}: %{y} options<extra></extra>")
])

add_left([
    go.Scatter(
        x=points_matrix[:, 0], y=points_matrix[:, 1], mode="markers",
        marker={"size": 7, "color": density, "colorscale": "RdBu", "cmid": 0,
                "cmin": -limit, "cmax": limit},
        hovertemplate="charge density=%{marker.color:.3g}<extra></extra>", showlegend=False,
    )
])

for label, start, stop, colour in BLOCKS:
    colours = ["#E5E7EB"] * V2_DIMENSIONS
    if start is not None:
        for index in range(start, stop):
            colours[index] = colour
    walk.add_trace(
        go.Bar(x=np.arange(V2_DIMENSIONS), y=vector, marker_color=colours, showlegend=False,
               hovertemplate="coordinate %{x}<br>value=%{y:.3f}<extra></extra>"),
        row=1, col=2,
    )

total_left = sum(left_counts)
steps = []
for step_index, (label, start, stop, _) in enumerate(BLOCKS):
    visible = [False] * (total_left + len(BLOCKS))
    offset = sum(left_counts[:step_index])
    for k in range(left_counts[step_index]):
        visible[offset + k] = True
    visible[total_left + step_index] = True
    written = "no coordinates yet" if start is None else f"coordinates {start}-{stop - 1} ({stop - start} values)"
    steps.append({
        "label": label.split(".")[0],
        "method": "update",
        "args": [{"visible": visible}, {"title": f"step {label}  ->  {written}"}],
    })

for index in range(total_left + len(BLOCKS)):
    walk.data[index].visible = False
for k in range(left_counts[0]):
    walk.data[k].visible = True
walk.data[total_left].visible = True

walk.update_yaxes(scaleanchor="x", scaleratio=1, row=1, col=1)
walk.update_xaxes(title_text="coordinate", row=1, col=2)
walk.update_yaxes(title_text="value", row=1, col=2)
walk.update_layout(
    title=f"step {BLOCKS[0][0]}  ->  no coordinates yet",
    template="plotly_white", height=560, bargap=0,
    sliders=[{"active": 0, "currentvalue": {"prefix": "step "}, "pad": {"t": 70}, "steps": steps}],
)
walk.show()

Three things are worth noticing as you step through.

**Each step reads a different view of the same geometry.** Step 3 reads a
distance transform, step 4 reads exact boundary distances, step 5 reads a
raster, step 7 reads a linear solve. None of them reads another design.

**The slices are disjoint and fixed.** Step 4 always writes coordinates 48
through 239, whatever the layout is. That is what makes coordinate 61 mean the
same thing for every contributor.

**Step 6 is the only one that does not look at the GDS file.** If a contributor
supplies no design options, coordinates 368-463 are zero and everything else is
unchanged. The geometry blocks never depend on the parameter mapping, which is
why Tutorial 18 can strip the parameter block out and still evaluate the encoder.

## 9. Assembling the vector

Five blocks, concatenated in a fixed order. No normalization against a
catalogue, no whitening, no learned projection - just the measurements.

In [17]:
# %% hide input
bounds = [
    ("physical metrics", 0, METRIC_BLOCK_SIZE, "#F4A261"),
    ("coupling spectrum", METRIC_BLOCK_SIZE, METRIC_BLOCK_SIZE + COUPLING_BLOCK_SIZE, "#00798C"),
    (
        "shape spectrum",
        METRIC_BLOCK_SIZE + COUPLING_BLOCK_SIZE,
        METRIC_BLOCK_SIZE + COUPLING_BLOCK_SIZE + SHAPE_BLOCK_SIZE,
        "#6A4C93",
    ),
    (
        "parameter statistics",
        METRIC_BLOCK_SIZE + COUPLING_BLOCK_SIZE + SHAPE_BLOCK_SIZE,
        V2_DIMENSIONS - PHYSICS_BLOCK_SIZE,
        "#E9C46A",
    ),
    ("physics proxy", V2_DIMENSIONS - PHYSICS_BLOCK_SIZE, V2_DIMENSIONS, "#2A9D8F"),
]
figure = go.Figure()
for name, start, stop, color in bounds:
    figure.add_trace(
        go.Bar(
            x=np.arange(start, stop),
            y=vector[start:stop],
            name=f"{name} ({stop - start})",
            marker_color=color,
            hovertemplate=f"<b>{name}</b><br>coordinate %{{x}}<br>value=%{{y:.3f}}<extra></extra>",
        )
    )
figure.update_layout(
    title=f"The complete universal-geometry-v2 vector for {SOURCE_ID}",
    xaxis_title="coordinate",
    yaxis_title="value",
    bargap=0,
    template="plotly_white",
    height=470,
)
figure.show()

occupancy = pd.DataFrame(
    [
        {
            "block": name,
            "dimensions": stop - start,
            "non-zero here": int(np.count_nonzero(vector[start:stop])),
        }
        for name, start, stop, _ in bounds
    ]
)
print(occupancy.to_string(index=False))
print(f"\ntotal: {V2_DIMENSIONS} dimensions, {int(np.count_nonzero(vector))} non-zero for this two-terminal device")

               block  dimensions  non-zero here
    physical metrics          48             42
   coupling spectrum         192             14
      shape spectrum         128             84
parameter statistics          96             58
       physics proxy          48             23

total: 512 dimensions, 221 non-zero for this two-terminal device


Roughly a third of the coordinates are non-zero. The empty ones are the
terminal-pair spectra for terminals this device does not have, and the parameter
classes it does not use. That reserved capacity costs nothing statistically - a
constant column contributes nothing to a standardized regression - and it is
what lets a four-terminal qubit-coupler share the vector layout with this
two-terminal capacitor.

## 10. Checking the invariances

The encoder claims a specific set of symmetries. Each one is checkable in a few
lines, so here they are, checked.

In [18]:
import klayout.db as kdb


def transformed_copy(path, destination, *, shift=(0.0, 0.0), scale=1.0, mirror=False, rotate=False):
    layout = kdb.Layout()
    layout.read(str(path))
    transform = kdb.DCplxTrans(scale, 270.0 if rotate else 0.0, mirror, shift[0], shift[1])
    for cell in layout.each_cell():
        for index in layout.layer_indices():
            cell.shapes(index).transform(transform)
    layout.write(str(destination))
    return destination


work = Path(os.getenv("TMPDIR", "/tmp")) / "squadds-tutorial19"
work.mkdir(parents=True, exist_ok=True)

coupling_span = slice(METRIC_BLOCK_SIZE, METRIC_BLOCK_SIZE + COUPLING_BLOCK_SIZE)
metric_span = slice(0, METRIC_BLOCK_SIZE)

checks = []
for label, kwargs in [
    ("translate by (4000, -2500) um", {"shift": (4000.0, -2500.0)}),
    ("mirror in x", {"mirror": True}),
    ("rotate by 270 degrees", {"rotate": True}),
    ("scale by 3x", {"scale": 3.0}),
]:
    variant = transformed_copy(GDS_PATH, work / f"{abs(hash(label))}.gds", **kwargs)
    moved = encode(variant, DESIGN_OPTIONS)
    checks.append(
        {
            "transform": label,
            "whole vector": round(float(np.abs(vector - moved).max()), 6),
            "coupling spectrum": round(float(np.abs(vector[coupling_span] - moved[coupling_span]).max()), 6),
            "physical metrics": round(float(np.abs(vector[metric_span] - moved[metric_span]).max()), 6),
        }
    )
invariance = pd.DataFrame(checks)
print(invariance.to_string(index=False))
print()
print("Read the last two columns separately: the coupling spectrum is a set of")
print("distances and is orientation free, while the metric block deliberately")
print("stores bbox width and height apart, so rotating a 11 x 31 um device swaps")
print("them. Only translation is invariant everywhere.")

                    transform  whole vector  coupling spectrum  physical metrics
translate by (4000, -2500) um      0.000000           0.000000          0.000000
                  mirror in x      0.003493           0.000028          0.003493
        rotate by 270 degrees      2.072184           0.000040          2.072184
                  scale by 3x      5.585610           3.916810          2.197196

Read the last two columns separately: the coupling spectrum is a set of
distances and is orientation free, while the metric block deliberately
stores bbox width and height apart, so rotating a 11 x 31 um device swaps
them. Only translation is invariant everywhere.


Translation is exact to the last bit, because the encoder re-origins the layout
on its conductor bounds before measuring anything.

Mirroring barely moves the vector. The residual is not a modelling choice but a
sampling artefact: reflecting the boundary changes which of the 1,024 samples
land near a bin centre, and soft binning keeps that perturbation small.

**Rotation deserves care, and the table corrects a tempting overstatement.** The
coupling spectrum is built from distances, so it is essentially untouched by a
270-degree rotation. The metric block is not, and should not be: it stores
`log1p_bbox_width_um` and `log1p_bbox_height_um` as separate coordinates, so
rotating an 11 by 31 micrometre device swaps them and moves those entries a long
way. v2 is orientation-free in the blocks that describe shape and coupling, and
orientation-aware in the block that describes extent. If a downstream task needs
full rotational invariance, drop or symmetrize the four extent coordinates rather
than assuming the whole vector already has it.

Scale is the last row and the important one. v0 and v1 are invariant here, which
sounds like a virtue and is actually the defect Tutorial 18 traced: a design and
its 3x copy have very different capacitance, so an encoder that maps them to the
same vector has destroyed the answer. v2 moves, and it moves in a structured way
- the whole coupling spectrum slides up by exactly $\log 3$ in separation.

## 11. The scenario this was all built for

Now the payoff. A group we have never met sends us one capacitor. It has a
different outline, a different finger style, a surrounding guard ring, and **28
parameters with names none of our tooling recognizes**.

We have never seen this design, never fitted anything to it, and have no
simulation result for it. We encode it with the same frozen function and ask the
catalogue which of our 13,683 designs it most resembles.

In [19]:
def build_foreign_capacitor(path):
    """A deliberately different two-terminal capacitor from an imaginary group."""
    layout = kdb.Layout()
    layout.dbu = 0.001
    top = layout.create_cell("TOP")

    def box(x0, y0, x1, y1):
        return kdb.Box(*[int(round(value * 1000)) for value in (x0, y0, x1, y1)])

    left = kdb.Region()
    right = kdb.Region()
    digits, pitch, digit_length = 9, 5.5, 26.0
    for index in range(digits):
        y = index * pitch
        # Tapered horizontal digits interleaved from opposite spines.
        target = left if index % 2 == 0 else right
        if index % 2 == 0:
            target.insert(box(-digit_length, y, 2.0, y + 2.2))
        else:
            target.insert(box(-2.0, y, digit_length, y + 2.2))
    left.insert(box(-digit_length - 4.0, -3.0, -digit_length, digits * pitch + 3.0))
    right.insert(box(digit_length, -3.0, digit_length + 4.0, digits * pitch + 3.0))
    conductors = left + right

    guard = kdb.Region(box(-digit_length - 16, -15, digit_length + 16, digits * pitch + 15))
    ground = guard - conductors.sized(9000)
    top.shapes(layout.layer(1, 0)).insert(ground)
    top.shapes(layout.layer(1, 10)).insert(conductors)
    top.shapes(layout.layer(2, 0)).insert(box(-digit_length - 4.0, digits * pitch + 3.0, -digit_length, digits * pitch + 5.0))
    top.shapes(layout.layer(3, 0)).insert(box(digit_length, -5.0, digit_length + 4.0, -3.0))
    layout.write(str(path))
    return path


FOREIGN_OPTIONS = {
    "digit_pitch": "5.5um",
    "digit_extent": "26um",
    "digit_thickness": "2.2um",
    "digit_population": 9,
    "spine_thickness": "4um",
    "guard_clearance": "9um",
    "guard_extent_x": "84um",
    "guard_extent_y": "74um",
    "substrate_index": 11.45,
    "metal_layer": 1,
    "is_symmetric": True,
    "launch_rotation": "0deg",
    **{f"aux_length_{index}": f"{1.0 + 0.7 * index}um" for index in range(10)},
    **{f"aux_flag_{index}": bool(index % 2) for index in range(6)},
}

foreign_path = build_foreign_capacitor(work / "foreign.gds")
foreign_vector = encode(foreign_path, FOREIGN_OPTIONS)
print(f"foreign design: {len(FOREIGN_OPTIONS)} parameters, none of which appear in our catalogue")
print(f"shared parameter names with our design: {sorted(set(FOREIGN_OPTIONS) & set(DESIGN_OPTIONS))}")
print(f"encoded to {foreign_vector.shape[0]} dimensions with no refitting")

foreign design: 28 parameters, none of which appear in our catalogue
shared parameter names with our design: []
encoded to 512 dimensions with no refitting


In [20]:
CACHE = Path(os.getenv("SQUADDS_TUTORIAL18_CACHE", Path.home() / ".cache/squadds/tutorial18"))
V2_TABLE = Path(os.getenv("SQUADDS_V2_TABLE", CACHE / "universal-geometry-v2.parquet"))
if not V2_TABLE.is_file():
    raise FileNotFoundError(f"Build the v2 catalogue first (see Tutorial 18); expected {V2_TABLE}.")

catalogue = pd.read_parquet(V2_TABLE).drop_duplicates("design_id").reset_index(drop=True)
catalogue_matrix = np.vstack(catalogue["embedding"].to_numpy()).astype(np.float32)
targets = {
    item["notes"]["source_id"]: item["sim_results"]["north_to_south"] for item in json.loads(DATABASE.read_text())
}

# Standardize in float64: in float32 the spread of a constant coordinate such as
# terminal_count does not evaluate to exactly zero, and the column survives the
# filter only to divide by zero a moment later.
reference = catalogue_matrix.astype(np.float64)
center = reference.mean(axis=0)
scale = reference.std(axis=0)
keep = scale > 1e-8
center, scale = center[keep], scale[keep]
standardized = (reference[:, keep] - center) / scale
query = (foreign_vector.astype(np.float64)[keep] - center) / scale
print(f"{int(keep.sum())} of {V2_DIMENSIONS} coordinates vary across the catalogue and define the metric")

unit = standardized / np.maximum(np.linalg.norm(standardized, axis=1, keepdims=True), 1e-12)
query_unit = query / max(np.linalg.norm(query), 1e-12)
similarity = unit @ query_unit
assert np.isfinite(similarity).all(), "similarity must be finite for every catalogue row"
best = np.argsort(similarity)[::-1][:8]

neighbours = pd.DataFrame(
    {
        "source_id": catalogue.loc[best, "source_id"].to_numpy(),
        "cosine": similarity[best].round(4),
        "min gap (um)": catalogue.loc[best, "minimum_pair_gap_um"].to_numpy().round(3),
        "simulated C(N,S) (fF)": [round(targets.get(name, float("nan")), 4) for name in catalogue.loc[best, "source_id"]],
    }
)
print(f"foreign design minimum conductor gap: {float(np.expm1(foreign_vector[METRIC_NAMES.index('log1p_minimum_pair_gap_um')])):.3f} um")
print()
print(neighbours.to_string(index=False))

204 of 512 coordinates vary across the catalogue and define the metric
foreign design minimum conductor gap: 3.300 um

        source_id  cosine  min gap (um)  simulated C(N,S) (fF)
q3d_cap/cap_00788  0.3059           4.0                 3.3439
q3d_cap/cap_01208  0.3044           4.0                 4.9917
q3d_cap/cap_01178  0.3001           4.0                 3.9506
q3d_cap/cap_00789  0.2978           4.0                 3.4451
q3d_cap/cap_00764  0.2896           6.0                 2.1625
q3d_cap/cap_01177  0.2890           4.0                 3.8188
q3d_cap/cap_01179  0.2869           4.0                 4.0610
q3d_cap/cap_00722  0.2851           4.0                 2.9697


In [21]:
# %% hide input
figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["The stranger's capacitor", "Where it lands in our catalogue"],
    horizontal_spacing=0.12,
    column_widths=[0.44, 0.56],
)
foreign_geometry = read_layer_geometry(foreign_path)
foreign_grouped = _role_geometry(foreign_geometry, None)
for role in ("domain", "conductor", "port"):
    for key, shape in foreign_grouped[role]:
        for trace in polygon_traces(shape, role, ROLE_COLORS[role], show=False):
            figure.add_trace(trace, row=1, col=1)

rng = np.random.default_rng(19)
shown = rng.choice(len(catalogue), size=min(4000, len(catalogue)), replace=False)
gap_values = catalogue["minimum_pair_gap_um"].to_numpy()
figure.add_trace(
    go.Scattergl(
        x=similarity[shown],
        y=gap_values[shown],
        mode="markers",
        marker={"size": 4, "opacity": 0.3, "color": "#C7CDD4"},
        name="catalogue",
        hovertemplate="cosine=%{x:.3f}<br>min gap=%{y:.2f} um<extra></extra>",
    ),
    row=1,
    col=2,
)
figure.add_trace(
    go.Scattergl(
        x=similarity[best],
        y=gap_values[best],
        mode="markers",
        marker={"size": 11, "color": "#D1495B", "symbol": "star"},
        name="nearest eight",
        hovertemplate="cosine=%{x:.3f}<br>min gap=%{y:.2f} um<extra></extra>",
    ),
    row=1,
    col=2,
)
figure.update_yaxes(scaleanchor="x", scaleratio=1, row=1, col=1)
figure.update_xaxes(title_text="x (um)", row=1, col=1)
figure.update_yaxes(title_text="y (um)", row=1, col=1)
figure.update_xaxes(title_text="cosine similarity to the foreign design", row=1, col=2)
figure.update_yaxes(title_text="minimum conductor gap (um)", row=1, col=2)
figure.update_layout(
    title="An unseen design, an unseen parameter schema, and the neighbours it selects",
    template="plotly_white",
    height=530,
)
figure.show()

The nearest neighbours are not chosen because a parameter called `finger_gap`
matched something - no name in the foreign schema matches ours. They are chosen
because the two designs put comparable amounts of metal at comparable
separations, which is the quantity that sets the capacitance.

That is the whole proposition. The stranger did not have to adopt our design
tool, our parameter names, or our component library. They had to emit a GDS file
with documented layer roles, and the rest is measurement.

## 12. Summary

| Step | Input | Output | Fitted to anything? |
| --- | --- | --- | --- |
| Roles and terminals | GDS polygons | ordered terminal list | no |
| Physical metrics | terminals, raster | 48 named scalars in um | no |
| Coupling spectrum | boundary samples, exact distances | 192 log-spaced bins | no |
| Shape spectrum | raster FFT, contour FFT | 128 coefficients | no |
| Parameter statistics | any option mapping | 96 typed coordinates | no |
| Physics proxy | boundary-element solve | 48 coordinates | no |

Nothing in that last column changes, which is the property the whole design
exists to guarantee. `encode` is a pure function: run it on one design or on a
hundred thousand, in our lab or in someone else's, and coordinate 61 is always
the facing boundary length between terminals 0 and 1 at a separation of about
1.7 micrometres.

## 13. Porting this to a different design family or a different simulation

Suppose you want to build an embedding for a component family this encoder has
never seen, with simulation outputs that are not capacitance. What actually has
to change?

### Nothing, if you only change the component

This is the common case and it is worth stating first. A new capacitor, coupler,
or qubit variant needs **no code change at all**. Terminals are found as
connected components rather than declared, parameters are typed by physical
dimension rather than matched by name, and every bin edge is an absolute length.
Call `encode(your.gds, your_options)` and you are in the same 512-dimensional
space. Tutorial 20 does exactly this for `CapNInterdigitalTee` and
`TransmonCross` without touching the encoder.

### The one thing you must supply: layer roles

The encoder has to know which polygons are conductor, which are etch, which are
port markers, and which are the simulation domain. If your GDS follows the
published SQuADDS layer semantics this is automatic. If it does not, pass the
mapping explicitly and nothing else changes:

```python
from squadds.layouts import encode

vector = encode(
    "foreign_device.gds",
    design_options,
    layer_roles={(10, 0): "conductor", (10, 1): "etch", (11, 0): "port", (1, 0): "domain"},
)
```

This is the entire input contract. Get it wrong and every downstream block is
measuring the wrong polygons, which is why it is the first thing to check when a
foreign layout produces a strange vector.

### Four things to check before trusting the result

**1. Terminal count.** v2 reserves four terminals, giving six pairs. A device
with five or more functional conductors will have the extras silently dropped by
the area-ranked ordering. Check `metadata["terminal_count"]` against what you
expect.

**2. Terminal ordering.** Ordering is by port marker first, then descending area.
A family with no port markers and two near-equal-area terminals can order
inconsistently between designs, which scrambles the per-pair blocks. Add port
markers on layers 2 through 5, or confirm the areas separate cleanly.

**3. Length scale.** All distance bins span 0.1 um to 1000 um. A junction-scale
feature below 0.1 um or a metre-scale resonator above 1000 um clips into the end
bins and stops being resolved. Compare your minimum gap and bounding box against
that range before assuming the spectrum is informative.

**4. Whether the physics proxy means anything.** The boundary-element block
solves a two-dimensional *electrostatic* problem. For an inductive or eigenmode
target it is not wrong, it is simply uninformative, and it will sit in the vector
as 48 coordinates of noise. It costs nothing statistically, but do not expect it
to help.

### What needs a new version of the standard

The dividing line is simple: **changing an input needs nothing, changing the
meaning of a coordinate needs a new version.**

| Change | Needs a new standard? |
| --- | --- |
| New component family, new parameter names | no |
| Custom layer-role mapping | no |
| New simulation target | no, targets never enter the vector |
| More than four terminals | yes, the block layout and dimension change |
| Different distance range or bin count | yes, coordinate 61 would stop meaning what it means today |
| Adding a magnetostatic or path-length block | yes, dimensions change |

If you cross into the "yes" column, publish it as `universal-geometry-v3` with
its own frozen schema rather than editing v2 in place. Existing vectors stay
valid and comparable, which is the entire point of freezing the contract.

### Different simulation or analysis results

The encoder never sees results, so a new analysis type needs no encoder change
whatsoever. What it needs is a **canonical target mapping**, and Tutorial 20
shows the recipe:

1. find the physically equivalent quantity across families - it named
   `north_to_south`, `top_to_bottom`, and `cross_to_claw` all as "mutual
   capacitance between the two functional conductors";
2. write down that mapping explicitly, one field name per family;
3. log-transform if the scales differ by decades across families, which they did;
4. only then run the transfer study.

For eigenmode data the analogous canonical target would be something like
resonant frequency and external quality factor, mapped per family. Note the
honest caveat from Tutorial 20: nothing yet demonstrates that a representation
tuned on electrostatic problems carries over to eigenmode quantities. The
encoder will happily produce vectors for `CavityClawRouteMeander`; whether they
are *useful* for predicting `cavity_frequency` is an open experiment, and the
physics-proxy block is the part most likely to need a companion.

**Where to go next**

- Tutorial 18 for the quantitative comparison against `static-shape-v0`.
- Tutorial 20 for the three-class cross-component study built on this encoder.
- `squadds/layouts/geometry_v2.py` for the encoder, and `universal_v2_schema()`
  for the machine-readable contract.